# 1. 시작

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## 1) Import

In [ ]:
import pandas as pd
import numpy as np
import math
import random
from collections import defaultdict, Counter
from tqdm import tqdm
import os

## 2) Data Load

In [ ]:
#train = pd.read_csv('/data/train.csv', encoding='utf-8-sig')
#test = pd.read_csv('/data/test.csv', encoding='utf-8-sig')
data_path = "C:/Users/seung/OneDrive/바탕 화면/2025_ML/data"

train_file_path = os.path.join(data_path, 'train.csv')
test_file_path = os.path.join(data_path, 'test.csv')
sample_submission_file_path = os.path.join(data_path, 'sample_submission.csv')

train = pd.read_csv(train_file_path, encoding='utf-8-sig')
test = pd.read_csv(test_file_path, encoding='utf-8-sig')

print("Train shape:", train.shape)
print("Test shape:", test.shape)

## 3) 함수

In [ ]:
'''
# 이 대회는 난독화 리뷰를 해석하는 거지만
# 난독화 패턴은 한국어 초성 중성 종성별로 확률적 변형을 룰베이스로 구현한 것 같음.
# 그래서 그걸 레벤슈타인을 통해 확률적으로 구현하고 역으로 해석하는 방식 씀.
'''

In [ ]:
###############################################
# A) 자모 분해/결합 유틸
###############################################
BASE_CODE = 0xAC00
CHOSUNG  = 21 * 28
JUNGSUNG = 28

CHOSUNG_LIST = [
    'ㄱ','ㄲ','ㄴ','ㄷ','ㄸ','ㄹ','ㅁ','ㅂ','ㅃ','ㅅ',
    'ㅆ','ㅇ','ㅈ','ㅉ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ'
]
JUNGSUNG_LIST = [
    'ㅏ','ㅐ','ㅑ','ㅒ','ㅓ','ㅔ','ㅕ','ㅖ','ㅗ','ㅘ',
    'ㅙ','ㅚ','ㅛ','ㅜ','ㅝ','ㅞ','ㅟ','ㅠ','ㅡ','ㅢ','ㅣ'
]
JONGSUNG_LIST = [
    '', 'ㄱ','ㄲ','ㄳ','ㄴ','ㄵ','ㄶ','ㄷ','ㄹ','ㄺ',
    'ㄻ','ㄼ','ㄽ','ㄾ','ㄿ','ㅀ','ㅁ','ㅂ','ㅄ','ㅅ',
    'ㅆ','ㅇ','ㅈ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ'
]

def decompose_syllable(ch):
    code = ord(ch) - BASE_CODE
    if code < 0 or code > (0xD7A3 - 0xAC00):
        return None
    cho  = code // CHOSUNG
    jung = (code % CHOSUNG) // JUNGSUNG
    jong = (code % CHOSUNG) % JUNGSUNG
    return (CHOSUNG_LIST[cho], JUNGSUNG_LIST[jung], JONGSUNG_LIST[jong])

def compose_syllable(cho, jung, jong):
    if cho not in CHOSUNG_LIST or jung not in JUNGSUNG_LIST or jong not in JONGSUNG_LIST:
        return None
    cho_i = CHOSUNG_LIST.index(cho)
    jung_i = JUNGSUNG_LIST.index(jung)
    jong_i = JONGSUNG_LIST.index(jong)
    code = BASE_CODE + (cho_i * CHOSUNG) + (jung_i * JUNGSUNG) + jong_i
    return chr(code)

def to_jamo_list(text):
    """문자열 -> [(초성,중성,종성) or (ch,'','')]"""
    result = []
    for ch in text:
        decomp = decompose_syllable(ch)
        if decomp is None:
            # 범위 밖
            result.append((ch,'',''))
        else:
            result.append(decomp)
    return result

def from_jamo_list(jamo_list):
    """[(cho,jung,jong)] -> 문자열"""
    result = []
    for (cho, jung, jong) in jamo_list:
        if (cho in CHOSUNG_LIST) and (jung in JUNGSUNG_LIST) and (jong in JONGSUNG_LIST):
            syl = compose_syllable(cho, jung, jong)
            if syl:
                result.append(syl)
            else:
                result.append(cho+jung+jong)
        else:
            if jung=='' and jong=='':
                result.append(cho)
            else:
                result.append(cho+jung+jong)
    return "".join(result)

###############################################
# B) 레벤슈타인 alignment
###############################################
def align_jamo_list_simple(src_list, tgt_list):
    n = len(src_list)
    m = len(tgt_list)
    dp = [[0]*(m+1) for _ in range(n+1)]

    for i in range(1, n+1):
        dp[i][0] = i
    for j in range(1, m+1):
        dp[0][j] = j

    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if src_list[i-1] == tgt_list[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )

    alignment = []
    i, j = n, m
    while i>0 or j>0:
        if i>0 and dp[i][j] == dp[i-1][j] + 1:
            alignment.append((src_list[i-1], None))
            i-=1
        elif j>0 and dp[i][j] == dp[i][j-1] + 1:
            alignment.append((None, tgt_list[j-1]))
            j-=1
        else:
            alignment.append((src_list[i-1], tgt_list[j-1]))
            i-=1
            j-=1
    alignment.reverse()
    return alignment

###############################################
# C) build_single_jamo_map
###############################################
def build_single_jamo_map(train):
    confusion_counter = Counter()
    for idx, row in tqdm(train.iterrows(), total=len(train), desc="build_single_jamo_map"):
        in_text  = row['input']
        out_text = row['output']
        s_jamos = to_jamo_list(in_text)
        t_jamos = to_jamo_list(out_text)
        alignment = align_jamo_list_simple(s_jamos, t_jamos)
        for (s,t) in alignment:
            if s is not None and t is not None:
                confusion_counter[(s,t)] += 1

    jamo_map = defaultdict(list)
    for (s,t), freq in confusion_counter.items():
        jamo_map[s].append((t, freq))

    for s in jamo_map:
        arr = jamo_map[s]
        arr.sort(key=lambda x:x[1], reverse=True)
        jamo_map[s] = arr
    return jamo_map

###############################################
# D) build_bigram_map_refined
###############################################
def build_bigram_map_refined(train):
    bigram_counter = Counter()

    for idx, row in tqdm(train.iterrows(), total=len(train), desc="build_bigram_map_refined"):
        in_text = row['input']
        out_text = row['output']

        in_tokens  = in_text.split()
        out_tokens = out_text.split()
        min_len = min(len(in_tokens), len(out_tokens))
        for i_tok in range(min_len):
            s_jamos = to_jamo_list(in_tokens[i_tok])
            t_jamos = to_jamo_list(out_tokens[i_tok])
            # 앞에 (None,'','') 붙여서 alignment
            s_jamos_aug = [(None,'','')] + s_jamos
            t_jamos_aug = [(None,'','')] + t_jamos

            alignment = align_jamo_list_simple(s_jamos_aug, t_jamos_aug)

            for j in range(len(alignment)-1):
                (s1,t1) = alignment[j]
                (s2,t2) = alignment[j+1]
                if s1 is None or s2 is None or t1 is None or t2 is None:
                    continue
                s_pair = (s1, s2)
                t_pair = (t1, t2)
                bigram_counter[(s_pair, t_pair)] += 1

    bigram_map = defaultdict(list)
    for (s_pair, t_pair), freq in bigram_counter.items():
        bigram_map[s_pair].append((t_pair, freq))

    for sp in bigram_map:
        arr = bigram_map[sp]
        arr.sort(key=lambda x:x[1], reverse=True)
        bigram_map[sp] = arr

    return bigram_map

###############################################
# E) 빅그램 + 단일자모 혼합 디코더
###############################################
# 사전에 있는 단어인지 확인하기 위한 집합 (전역 변수 혹은 함수 인자로 전달)
# main_process 함수 내에서 vocab_set을 만들어서 넘겨줄 예정
def decode_word_rerank(src_word, bigram_map, single_map, vocab_set,
                       lam=0.1, topk=5, beam_size=10, threshold=3.0): # threshold 인자 추가
    
    s_jamos = to_jamo_list(src_word)
    s_jamos = [(None,'','')] + s_jamos
    n = len(s_jamos)

    beam = [ ([ (None,'','') ], (None,'',''), 0.0) ]

    for i in range(1, n):
        s_pair = (s_jamos[i-1], s_jamos[i])
        
        bigram_candidates = bigram_map.get(s_pair, [])[:topk]
        single_candidates = single_map.get(s_jamos[i], [])[:topk]

        new_beam = []
        for (decoded_list, last_jamo, sc) in beam:
            # 1) Bigram
            for (t_pair, freq_bi) in bigram_candidates:
                t_im1, t_i = t_pair
                if t_im1 == last_jamo:
                    freq_single = 0
                    if s_jamos[i] in single_map:
                        for (cand_t, f_s) in single_map[s_jamos[i]]:
                            if cand_t == t_i:
                                freq_single = f_s
                                break
                    step_score = math.log(1+freq_bi) + lam * math.log(1+freq_single)
                    new_beam.append((decoded_list + [t_i], t_i, sc + step_score))

            # 2) Single (Back-off)
            for (t_i, freq_single) in single_candidates:
                step_score = lam * math.log(1+freq_single)
                new_beam.append((decoded_list + [t_i], t_i, sc + step_score))

        new_beam.sort(key=lambda x: x[2], reverse=True)
        beam = new_beam[:beam_size]
        if not beam: break

    # --- Re-ranking 로직 ---
    candidates = []
    for (decoded_list, _, sc) in beam:
        filtered = [j for j in decoded_list if j[0] is not None]
        word = from_jamo_list(filtered)
        candidates.append((word, sc))
    
    if not candidates: return src_word

    best_word, best_score = candidates[0]
    if best_word in vocab_set: return best_word
    
    # threshold 파라미터 사용
    for i in range(1, len(candidates)):
        cand_word, cand_score = candidates[i]
        if best_score - cand_score > threshold: break
        if cand_word in vocab_set: return cand_word
            
    return best_word

def convert_review_bigram_mixed(line, bigram_map, single_map, word_freq,
                                lam=0.3, topk=5, beam_size=10):
    tokens = line.split()
    new_tokens = []
    for t in tokens:
        w = decode_word_bigram_mixed(t, bigram_map, single_map, word_freq,
                                     lam=lam, topk=topk, beam_size=beam_size)
        new_tokens.append(w)
    return " ".join(new_tokens)


def evaluate_on_train(train_df, bigram_map, single_map, vocab_set,
                      lam=0.1, topk=160, beam_size=160, threshold=3.0, 
                      sample_n=500):
    
    print(f"\n🔍 [Self-Check] 훈련 데이터 {sample_n}개 성능 검증 시작 (Threshold={threshold})...")
    
    # 훈련 데이터에서 랜덤 샘플링 (전체는 오래 걸리므로)
    sample_df = train_df.sample(n=min(sample_n, len(train_df)), random_state=42)
    
    total_words = 0
    correct_words = 0
    total_sentences = 0
    correct_sentences = 0
    
    # 검증 루프
    for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Evaluating"):
        input_text = row['input']
        target_text = row['output']
        
        tokens = input_text.split()
        target_tokens = target_text.split()
        
        restored_tokens = []
        for t in tokens:
            # 위에서 수정한 decode_word_rerank 호출
            restored_w = decode_word_rerank(t, bigram_map, single_map, vocab_set,
                                            lam=lam, topk=topk, beam_size=beam_size,
                                            threshold=threshold)
            restored_tokens.append(restored_w)
            
        # 정확도 계산 (단어 단위)
        # 띄어쓰기가 보존된다고 가정하고 1:1 비교
        min_len = min(len(restored_tokens), len(target_tokens))
        for i in range(min_len):
            if restored_tokens[i] == target_tokens[i]:
                correct_words += 1
            total_words += 1
            
        # 정확도 계산 (문장 단위)
        if " ".join(restored_tokens) == target_text:
            correct_sentences += 1
        total_sentences += 1
        
    word_acc = (correct_words / total_words * 100) if total_words > 0 else 0
    sent_acc = (correct_sentences / total_sentences * 100) if total_sentences > 0 else 0
    
    print(f"검증 결과")
    print(f" - 단어 정확도 (Word Acc): {word_acc:.2f}%")
    print(f" - 문장 정확도 (Sent Acc): {sent_acc:.2f}%")
    return word_acc

###############################################
# F) 전체 파이프라인
###############################################
def main_process_rerank(train, test, lam=0.1, topk=160, beam_size=160, threshold=3.0): # threshold 인자 추가
    print("[1] Maps building...")
    single_map = build_single_jamo_map(train)
    bigram_map = build_bigram_map_refined(train)

    print("[2] Vocab building...")
    vocab_set = set()
    for txt in train['output']:
        for w in txt.split():
            vocab_set.add(w)
            
    # ==========================================
    # [추가됨] 훈련 데이터 성능 검증 실행
    # ==========================================
    evaluate_on_train(train, bigram_map, single_map, vocab_set,
                      lam=lam, topk=topk, beam_size=beam_size, threshold=threshold,
                      sample_n=500) # 500개만 테스트
    # ==========================================

    print("[3] Decoding with Re-ranking...")
    outputs = []
    for idx, row in tqdm(test.iterrows(), total=len(test)):
        obf_text = row['input']
        tokens = obf_text.split()
        new_tokens = []
        
        for t in tokens:
            # Re-ranking 함수 호출
            restored = decode_word_rerank(t, bigram_map, single_map, vocab_set,
                                          lam=lam, topk=topk, beam_size=beam_size)
            new_tokens.append(restored)
            
        outputs.append(" ".join(new_tokens))

    test_result = test.copy()
    test_result['output'] = outputs
    return test_result

In [ ]:
## 4)추론

In [ ]:
result_df = main_process_rerank(train, test,
                                      lam=0.1, topk=160, beam_size=160)
result_df.head()

In [ ]:
## 5)제출

In [ ]:
# 1. sample_submission.csv 파일 불러오기
#submission_df = pd.read_csv('/data/sample_submission.csv')
submission_df = pd.read_csv(sample_submission_file_path, encoding='utf-8-sig')

# 2. result_df의 'output' 컬럼 내용을 submission_df의 'output' 컬럼에 덮어씌우기
submission_df['output'] = result_df['output']

# 3. submission50.csv 파일로 저장하기
submission_df.to_csv('submissionxx3.csv', index=False)

print("파일이 저장되었습니다.")